# Fine-Tuning Legal-BERT for Clause Classification

## Objective
Fine-tune a transformer-based model on legal contract clauses
to improve classification accuracy over baseline ML models.


In [ ]:
!pip install -q transformers datasets torch scikit-learn


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
dataset = load_dataset(
    "dvgodoy/CUAD_v1_Contract_Understanding_clause_classification"
)

dataset


In [ ]:
# Keep only top 8 most frequent clause types
top_labels = (
    dataset["train"]
    .to_pandas()["label"]
    .value_counts()
    .head(8)
    .index
    .tolist()
)

dataset = dataset.filter(lambda x: x["label"] in top_labels)

print("Labels used:", set(dataset["train"]["label"]))


In [ ]:
labels = sorted(list(set(dataset["train"]["label"])))

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)


In [ ]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clause"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["clause"])
tokenized_ds.set_format("torch")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


In [ ]:
# %pip install -q accelerate>=0.26.0

training_args = TrainingArguments(
    output_dir="../models/finetuned_clause_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)


In [12]:
# Split train into train and validation sets
train_val_split = tokenized_ds["train"].train_test_split(test_size=0.2, seed=42)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_val_split["train"],
    eval_dataset=train_val_split["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\ADITYA\AppData\Local\Temp\ipykernel_21632\1902079343.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.042500,0.039955,0.993263,0.993284
2,0.022200,0.029986,0.993263,0.993285
3,0.018500,0.027363,0.994012,0.994017


C:\Users\ADITYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\ADITYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=2004, training_loss=0.09739179618051555, metrics={'train_runtime': 11588.2201, 'train_samples_per_second': 1.383, 'train_steps_per_second': 0.173, 'total_flos': 2108422464528384.0, 'train_loss': 0.09739179618051555, 'epoch': 3.0})

In [14]:
trainer.save_model("../models/finetuned_clause_classifier")
tokenizer.save_pretrained("../models/finetuned_clause_classifier")


('../models/finetuned_clause_classifier\\tokenizer_config.json',
 '../models/finetuned_clause_classifier\\special_tokens_map.json',
 '../models/finetuned_clause_classifier\\vocab.txt',
 '../models/finetuned_clause_classifier\\added_tokens.json',
 '../models/finetuned_clause_classifier\\tokenizer.json')

In [15]:
model.save_pretrained("../models/finetuned_clause_classifier")
tokenizer.save_pretrained("../models/finetuned_clause_classifier")


('../models/finetuned_clause_classifier\\tokenizer_config.json',
 '../models/finetuned_clause_classifier\\special_tokens_map.json',
 '../models/finetuned_clause_classifier\\vocab.txt',
 '../models/finetuned_clause_classifier\\added_tokens.json',
 '../models/finetuned_clause_classifier\\tokenizer.json')